# Track A PR2 Fast 070202

PR2만 빠르게 재실행하기 위한 노트북이다. 원본 `trackA_pr_ablation_070202.ipynb`에서 PR0 baseline inference와 PR1 retraining을 제외하고, `top50 fold-balanced` 후보 제거 + duplicate down-weighting PR2 학습/제출 저장만 남겼다.

기본 `EPOCHS=10`은 기존 ablation과 비교 가능하게 유지했다. 빠른 smoke test만 필요하면 config 셀에서 `EPOCHS`를 1-3으로 낮춘다.

## 0. Imports and Seed


In [1]:
from pathlib import Path
import json
import random
import time
from datetime import datetime
from itertools import combinations

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from torchvision.models import ResNet18_Weights

try:
    from torchinfo import summary
except Exception:
    summary = None

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score

SEED = 42


def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))


device: cuda
gpu: NVIDIA GeForce RTX 4070 SUPER


## 1. Paths and PR2 Fast Config

In [2]:
PROJECT_DIR = Path.cwd()
if PROJECT_DIR.name != "Quest02":
    PROJECT_DIR = Path("/home/thkim0/github/AIFFEL_quest_rs/MainQuest/Quest02")

DATA_DIR = PROJECT_DIR / "data" / "RS18A"
TRAIN_DIR = DATA_DIR / "train"
TEST_DIR = DATA_DIR / "test"
TRAIN_LABELS_PATH = DATA_DIR / "train_labels.csv"
SAMPLE_SUBMISSION_PATH = DATA_DIR / "sample_submission.csv"

PR0_CHECKPOINT_PATH = PROJECT_DIR / "model" / "base0_resnet18_finetune_nocrop_best.pt"
PR0_HISTORY_PATH = PROJECT_DIR / "model" / "base0_resnet18_finetune_nocrop_history.csv"
PR0_SUBMISSION_SOURCE_PATH = PROJECT_DIR / "submissionA_resnet18_finetune_nocrop.csv"

OOF_DIR = PROJECT_DIR / "outputs" / "trackB_oof_mislabel"
OOF_SCORE_PATH = OOF_DIR / "oof_mislabel_scores.csv"
OOF_CONFIG_PATH = OOF_DIR / "trackB_oof_mislabel_config.json"
OOF_FOLD_ASSIGNMENT_PATH = OOF_DIR / "fold_assignments.csv"
OOF_FOLD_HISTORY_PATH = OOF_DIR / "fold_train_history.csv"
OOF_CHECKPOINT_DIR = OOF_DIR / "checkpoints"

V2_DETAIL_CANDIDATES = [
    PROJECT_DIR / "outputs" / "trackB_070103_v2_20260701_223101" / "trackB_v2_scores_detail.csv",
    PROJECT_DIR / "outputs" / "trackB_070103_v2" / "trackB_v2_scores_detail.csv",
]
V2_DETAIL_PATH = next((path for path in V2_DETAIL_CANDIDATES if path.exists()), None)

RUN_NAME = "trackA_pr2_fast_top50_fold_balanced_" + datetime.now().strftime("%Y%m%d_%H%M%S")
OUTPUT_DIR = PROJECT_DIR / "outputs" / RUN_NAME
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
TABLE_DIR = OUTPUT_DIR / "tables"
VIS_DIR = OUTPUT_DIR / "visualizations"
SUBMISSION_DIR = OUTPUT_DIR / "submissions"
for path in [OUTPUT_DIR, CHECKPOINT_DIR, TABLE_DIR, VIS_DIR, SUBMISSION_DIR]:
    path.mkdir(parents=True, exist_ok=True)

VAL_RATIO = 0.2
BATCH_SIZE = 16
EPOCHS = 10
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 0
PIN_MEMORY = torch.cuda.is_available()

ACTIVE_CANDIDATE_NAME = "candidate_top50_fold_balanced"
CANDIDATE_TOP50_PER_FOLD = 10
CANDIDATE_TOP100_PER_FOLD = 20
TOPK_REPORTS = [50, 100]
EXPECTED_OOF_CHECKPOINT = "best_holdout_auc"
REBUILD_BEST_AUC_OOF_IF_NEEDED = True
REQUIRE_BEST_AUC_OOF = True

DUP_WEIGHT_MIN = 0.50
DUP_WEIGHT_POWER = 1.0
RUN_PR1 = False
RUN_PR2 = True

CONFIG_PATH = OUTPUT_DIR / "trackA_pr2_fast_config.json"
config = {
    "seed": SEED,
    "run_name": RUN_NAME,
    "data_dir": str(DATA_DIR),
    "pr0_checkpoint_path": str(PR0_CHECKPOINT_PATH),
    "pr0_history_path": str(PR0_HISTORY_PATH),
    "pr0_submission_source_path": str(PR0_SUBMISSION_SOURCE_PATH),
    "pr0_transform": "Resize((224, 224)) -> ToTensor -> ImageNet Normalize",
    "oof_score_path": str(OOF_SCORE_PATH),
    "oof_config_path": str(OOF_CONFIG_PATH),
    "expected_oof_checkpoint": EXPECTED_OOF_CHECKPOINT,
    "rebuild_best_auc_oof_if_needed": REBUILD_BEST_AUC_OOF_IF_NEEDED,
    "require_best_auc_oof": REQUIRE_BEST_AUC_OOF,
    "active_candidate_name": ACTIVE_CANDIDATE_NAME,
    "candidate_top50_per_fold": CANDIDATE_TOP50_PER_FOLD,
    "candidate_top100_per_fold": CANDIDATE_TOP100_PER_FOLD,
    "val_ratio": VAL_RATIO,
    "batch_size": BATCH_SIZE,
    "epochs": EPOCHS,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "dup_weight_min": DUP_WEIGHT_MIN,
    "dup_weight_power": DUP_WEIGHT_POWER,
    "output_dir": str(OUTPUT_DIR),
    "fast_notebook_mode": "PR2 only; PR0/PR1 training skipped",
}
with CONFIG_PATH.open("w", encoding="utf-8") as f:
    json.dump(config, f, indent=2, ensure_ascii=False)

required_paths = [
    DATA_DIR,
    TRAIN_DIR,
    TEST_DIR,
    TRAIN_LABELS_PATH,
    SAMPLE_SUBMISSION_PATH,
    PR0_CHECKPOINT_PATH,
    PR0_HISTORY_PATH,
    PR0_SUBMISSION_SOURCE_PATH,
    OOF_SCORE_PATH,
    OOF_CONFIG_PATH,
    OOF_FOLD_ASSIGNMENT_PATH,
]
for path in required_paths:
    if not path.exists():
        raise FileNotFoundError(path)

if V2_DETAIL_PATH is None:
    raise FileNotFoundError(f"No V2 detail candidate found: {V2_DETAIL_CANDIDATES}")

print("PROJECT_DIR:", PROJECT_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("V2_DETAIL_PATH:", V2_DETAIL_PATH)
print("saved config:", CONFIG_PATH)

print("Fast mode: PR2 only. Set EPOCHS lower in this cell for a smoke test if needed.")


PROJECT_DIR: /home/thkim0/github/AIFFEL_quest_rs/MainQuest/Quest02
OUTPUT_DIR: /home/thkim0/github/AIFFEL_quest_rs/MainQuest/Quest02/outputs/trackA_pr2_fast_top50_fold_balanced_20260702_212301
V2_DETAIL_PATH: /home/thkim0/github/AIFFEL_quest_rs/MainQuest/Quest02/outputs/trackB_070103_v2_20260701_223101/trackB_v2_scores_detail.csv
saved config: /home/thkim0/github/AIFFEL_quest_rs/MainQuest/Quest02/outputs/trackA_pr2_fast_top50_fold_balanced_20260702_212301/trackA_pr2_fast_config.json
Fast mode: PR2 only. Set EPOCHS lower in this cell for a smoke test if needed.


## 2. Data, Transform, Model Utilities


In [3]:
IMAGE_EXTENSIONS = [".jpg", ".jpeg", ".png", ".bmp", ".webp"]


def image_path_from_id(image_dir, image_id):
    image_id = str(image_id)
    for ext in IMAGE_EXTENSIONS:
        candidate = Path(image_dir) / f"{image_id}{ext}"
        if candidate.exists():
            return str(candidate)
    raise FileNotFoundError(f"Image not found for id={image_id} in {image_dir}")


weights = ResNet18_Weights.IMAGENET1K_V1
imagenet_mean = weights.transforms().mean
imagenet_std = weights.transforms().std

# PR0 nocrop/prep checkpoint was trained with direct 224x224 resize, not CenterCrop.
base_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std),
])


class TrainWeightedDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True).copy()
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row["path"]).convert("RGB")
        if self.transform is not None:
            image = self.transform(image)
        return image, int(row["label"]), float(row["sample_weight"])


class EvalDataset(Dataset):
    def __init__(self, dataframe, transform=None, has_label=True):
        self.df = dataframe.reset_index(drop=True).copy()
        self.transform = transform
        self.has_label = has_label

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row["path"]).convert("RGB")
        if self.transform is not None:
            image = self.transform(image)
        if self.has_label:
            return image, int(row["label"]), str(row["id"])
        return image, str(row["id"])


def build_resnet18(num_classes=2, pretrained=True):
    model_weights = weights if pretrained else None
    model = models.resnet18(weights=model_weights)
    in_features = model.fc.in_features
    model.fc = nn.Linear(in_features, num_classes)
    for param in model.parameters():
        param.requires_grad = True
    return model


def make_train_loader(dataframe, shuffle=True):
    return DataLoader(
        TrainWeightedDataset(dataframe, transform=base_transform),
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
    )


def make_eval_loader(dataframe, has_label=True):
    return DataLoader(
        EvalDataset(dataframe, transform=base_transform, has_label=has_label),
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
    )


def safe_auc(labels, probs):
    labels = np.asarray(labels)
    probs = np.asarray(probs)
    if len(labels) == 0 or len(np.unique(labels)) < 2:
        return np.nan
    return float(roc_auc_score(labels, probs))


def print_trainable_parameters(model, run_name):
    rows = []
    for name, param in model.named_parameters():
        if param.requires_grad:
            rows.append({
                "run": run_name,
                "name": name,
                "shape": str(tuple(param.shape)),
                "num_params": int(param.numel()),
            })
    df = pd.DataFrame(rows)
    print(f"[{run_name}] trainable parameter count:", int(df["num_params"].sum()) if len(df) else 0)
    display(df)
    return df


## 3. Load Labels and Reproduce Baseline Split


In [4]:
train_df = pd.read_csv(TRAIN_LABELS_PATH)
submission_template = pd.read_csv(SAMPLE_SUBMISSION_PATH)

expected_label_cols = ["id", "label"]
if train_df.columns.tolist() != expected_label_cols:
    raise ValueError(f"train_labels columns mismatch: {train_df.columns.tolist()}")
if len(train_df) != 1366:
    raise ValueError(f"train label row count must be 1366, got {len(train_df)}")
if train_df["id"].duplicated().any():
    raise ValueError("train_labels has duplicated ids")
if sorted(train_df["label"].unique().tolist()) != [0, 1]:
    raise ValueError("label must be binary 0/1")

train_df["id"] = train_df["id"].astype(str)
train_df["label"] = train_df["label"].astype(int)
train_df["path"] = train_df["id"].map(lambda image_id: image_path_from_id(TRAIN_DIR, image_id))

submission_template["id"] = submission_template["id"].astype(str)
test_df = submission_template[["id"]].copy()
test_df["path"] = test_df["id"].map(lambda image_id: image_path_from_id(TEST_DIR, image_id))

base_train_df, base_val_df = train_test_split(
    train_df,
    test_size=VAL_RATIO,
    random_state=SEED,
    stratify=train_df["label"],
)
base_train_df = base_train_df.reset_index(drop=True).copy()
base_val_df = base_val_df.reset_index(drop=True).copy()

split_df = pd.concat([
    base_train_df.assign(baseline_split="train"),
    base_val_df.assign(baseline_split="val"),
], ignore_index=True)

if len(split_df) != len(train_df) or split_df["id"].nunique() != len(train_df):
    raise ValueError("PR0 split does not cover train_df exactly once")

split_df.to_csv(TABLE_DIR / "pr0_split_assignments.csv", index=False)

print("full label distribution:")
display(train_df["label"].value_counts().sort_index().rename_axis("label").reset_index(name="count"))
print("PR0 split label distribution:")
display(pd.crosstab(split_df["baseline_split"], split_df["label"]))
print("saved PR0 split assignments:", TABLE_DIR / "pr0_split_assignments.csv")


full label distribution:


,label,count
0,0,813
1,1,553


PR0 split label distribution:


label,0,1
baseline_split,,
train,650,442
val,163,111


saved PR0 split assignments: /home/thkim0/github/AIFFEL_quest_rs/MainQuest/Quest02/outputs/trackA_pr2_fast_top50_fold_balanced_20260702_212301/tables/pr0_split_assignments.csv


## 4. Validate OOF File and Rebuild Best-AUC OOF if Needed


In [5]:
oof_config = json.loads(OOF_CONFIG_PATH.read_text())
oof_predict_checkpoint = oof_config.get("oof_predict_checkpoint")
print("OOF config checkpoint:", oof_predict_checkpoint)
print("OOF config n_splits:", oof_config.get("n_splits"), "epochs:", oof_config.get("epochs"))

raw_oof_df = pd.read_csv(OOF_SCORE_PATH)
raw_oof_df["id"] = raw_oof_df["id"].astype(str)
required_oof_cols = ["id", "path", "label", "fold", "oof_dusty_prob", "oof_pred", "mislabel_score_oof"]
missing_cols = [col for col in required_oof_cols if col not in raw_oof_df.columns]
if missing_cols:
    raise ValueError(f"OOF missing columns: {missing_cols}")
if len(raw_oof_df) != 1366:
    raise ValueError(f"OOF row count must be 1366, got {len(raw_oof_df)}")
if raw_oof_df["id"].nunique() != 1366:
    raise ValueError("OOF ids must be unique and cover 1366 samples")
if raw_oof_df[required_oof_cols].isna().any().any():
    raise ValueError("OOF required columns contain NaN")
if not raw_oof_df["oof_dusty_prob"].between(0, 1).all():
    raise ValueError("oof_dusty_prob must be in [0, 1]")

fold_assignment_df = pd.read_csv(OOF_FOLD_ASSIGNMENT_PATH)
fold_assignment_df["id"] = fold_assignment_df["id"].astype(str)
fold_assignment_df["label"] = fold_assignment_df["label"].astype(int)
fold_assignment_df["fold"] = fold_assignment_df["fold"].astype(int)
fold_assignment_df["path"] = fold_assignment_df["id"].map(lambda image_id: image_path_from_id(TRAIN_DIR, image_id))

if len(fold_assignment_df) != 1366 or fold_assignment_df["id"].nunique() != 1366:
    raise ValueError("fold assignments must cover 1366 unique ids")
if sorted(fold_assignment_df["fold"].unique().tolist()) != list(range(int(oof_config.get("n_splits", 5)))):
    raise ValueError("fold assignment values do not match OOF config")

checkpoint_audit_rows = []
fold_history_df = pd.read_csv(OOF_FOLD_HISTORY_PATH) if OOF_FOLD_HISTORY_PATH.exists() else pd.DataFrame()
for fold in sorted(fold_assignment_df["fold"].unique()):
    best_path = OOF_CHECKPOINT_DIR / f"fold{fold}_best_holdout_auc.pt"
    final_path = OOF_CHECKPOINT_DIR / f"fold{fold}_final.pt"
    best_epoch = np.nan
    best_auc = np.nan
    if best_path.exists():
        checkpoint = torch.load(best_path, map_location="cpu")
        best_epoch = checkpoint.get("epoch", np.nan)
        best_auc = checkpoint.get("best_holdout_auc", np.nan)
    history_best_epoch = np.nan
    history_best_auc = np.nan
    if not fold_history_df.empty:
        part = fold_history_df.loc[fold_history_df["fold"] == fold].copy()
        if len(part):
            idx = part["holdout_auc"].astype(float).idxmax()
            history_best_epoch = int(part.loc[idx, "epoch"])
            history_best_auc = float(part.loc[idx, "holdout_auc"])
    checkpoint_audit_rows.append({
        "fold": fold,
        "best_checkpoint_path": str(best_path),
        "best_checkpoint_exists": best_path.exists(),
        "final_checkpoint_exists": final_path.exists(),
        "checkpoint_best_epoch": best_epoch,
        "checkpoint_best_auc": best_auc,
        "history_best_epoch": history_best_epoch,
        "history_best_auc": history_best_auc,
    })

checkpoint_audit_df = pd.DataFrame(checkpoint_audit_rows)
checkpoint_audit_df.to_csv(TABLE_DIR / "oof_checkpoint_audit.csv", index=False)
display(checkpoint_audit_df)

best_checkpoint_ready = bool(checkpoint_audit_df["best_checkpoint_exists"].all())
oof_is_best_auc = oof_predict_checkpoint == EXPECTED_OOF_CHECKPOINT
print("OOF file claims best-AUC checkpoint:", oof_is_best_auc)
print("best checkpoint files ready:", best_checkpoint_ready)


def predict_labeled_dataframe(model, dataframe):
    loader = make_eval_loader(dataframe, has_label=True)
    model.eval()
    rows = []
    with torch.no_grad():
        for images, labels, image_ids in loader:
            images = images.to(device, non_blocking=PIN_MEMORY)
            logits = model(images)
            probs = torch.softmax(logits, dim=1)[:, 1].detach().cpu().numpy()
            preds = logits.argmax(dim=1).detach().cpu().numpy()
            labels_np = labels.detach().cpu().numpy()
            for image_id, label, pred, prob in zip(image_ids, labels_np, preds, probs):
                rows.append({
                    "id": str(image_id),
                    "label": int(label),
                    "oof_pred": int(pred),
                    "oof_dusty_prob": float(prob),
                })
    return pd.DataFrame(rows)


def rebuild_oof_from_best_checkpoints():
    pred_parts = []
    for fold in sorted(fold_assignment_df["fold"].unique()):
        checkpoint_path = OOF_CHECKPOINT_DIR / f"fold{fold}_best_holdout_auc.pt"
        if not checkpoint_path.exists():
            raise FileNotFoundError(checkpoint_path)
        holdout_df = fold_assignment_df.loc[fold_assignment_df["fold"] == fold].copy()
        model = build_resnet18(num_classes=2, pretrained=False).to(device)
        checkpoint = torch.load(checkpoint_path, map_location=device)
        load_result = model.load_state_dict(checkpoint["model_state_dict"], strict=False)
        if load_result.missing_keys or load_result.unexpected_keys:
            raise ValueError(
                f"fold {fold} checkpoint key mismatch: "
                f"missing={load_result.missing_keys}, unexpected={load_result.unexpected_keys}"
            )
        pred_df = predict_labeled_dataframe(model, holdout_df)
        pred_df["fold"] = fold
        pred_df["checkpoint_path"] = str(checkpoint_path)
        pred_df["checkpoint_epoch"] = checkpoint.get("epoch", np.nan)
        pred_df["checkpoint_best_holdout_auc"] = checkpoint.get("best_holdout_auc", np.nan)
        pred_parts.append(pred_df)
        del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    rebuilt = pd.concat(pred_parts, ignore_index=True)
    rebuilt = fold_assignment_df[["id", "path", "label", "fold"]].merge(
        rebuilt.drop(columns=["label", "fold"]),
        on="id",
        how="left",
    )
    rebuilt["mislabel_score_oof"] = np.where(
        rebuilt["label"].astype(int) == 0,
        rebuilt["oof_dusty_prob"].astype(float),
        1.0 - rebuilt["oof_dusty_prob"].astype(float),
    )
    for col in ["dup_score", "mislabel_score_v2_add"]:
        if col in raw_oof_df.columns:
            rebuilt = rebuilt.merge(raw_oof_df[["id", col]], on="id", how="left")
    return rebuilt


if oof_is_best_auc:
    oof_df = raw_oof_df.copy()
    oof_source = "existing_best_auc_oof_file"
elif REBUILD_BEST_AUC_OOF_IF_NEEDED and best_checkpoint_ready:
    print("Existing OOF file is not best-AUC based. Rebuilding OOF predictions from fold best checkpoints...")
    oof_df = rebuild_oof_from_best_checkpoints()
    oof_source = "rebuilt_from_best_auc_checkpoints"
    rebuilt_path = TABLE_DIR / "oof_mislabel_scores_best_auc_rebuilt.csv"
    oof_df.to_csv(rebuilt_path, index=False)
    print("saved rebuilt best-AUC OOF:", rebuilt_path)
else:
    oof_df = raw_oof_df.copy()
    oof_source = "existing_non_best_oof_file"

if REQUIRE_BEST_AUC_OOF and oof_source == "existing_non_best_oof_file":
    raise ValueError(
        "OOF file is not best-AUC based and best checkpoints could not be used to rebuild it. "
        "Set REQUIRE_BEST_AUC_OOF=False only for a diagnostic dry run."
    )

print("OOF source used by this notebook:", oof_source)


OOF config checkpoint: final
OOF config n_splits: 5 epochs: 10


,fold,best_checkpoint_path,best_checkpoint_exists,final_checkpoint_exists,checkpoint_best_epoch,checkpoint_best_auc,history_best_epoch,history_best_auc
0,0,/home/thkim0/github/AIFFEL_quest_rs/MainQuest/...,True,True,9,0.774001,9,0.774001
1,1,/home/thkim0/github/AIFFEL_quest_rs/MainQuest/...,True,True,10,0.751813,10,0.751813
2,2,/home/thkim0/github/AIFFEL_quest_rs/MainQuest/...,True,True,7,0.800725,7,0.800725
3,3,/home/thkim0/github/AIFFEL_quest_rs/MainQuest/...,True,True,1,0.820209,1,0.820209
4,4,/home/thkim0/github/AIFFEL_quest_rs/MainQuest/...,True,True,4,0.794406,4,0.794406


OOF file claims best-AUC checkpoint: False
best checkpoint files ready: True
Existing OOF file is not best-AUC based. Rebuilding OOF predictions from fold best checkpoints...


/home/thkim0/venv/v1/lib/python3.10/site-packages/torch/nn/modules/conv.py:459: UserWarning: Applied workaround for CuDNN issue, install nvrtc.so (Triggered internally at ../aten/src/ATen/native/cudnn/Conv_v8.cpp:80.)
  return F.conv2d(input, weight, bias, self.stride,


saved rebuilt best-AUC OOF: /home/thkim0/github/AIFFEL_quest_rs/MainQuest/Quest02/outputs/trackA_pr2_fast_top50_fold_balanced_20260702_212301/tables/oof_mislabel_scores_best_auc_rebuilt.csv
OOF source used by this notebook: rebuilt_from_best_auc_checkpoints


## 5. Compute Mislabel Score and Compare with v2


In [6]:
oof_df = oof_df.copy()
oof_df["id"] = oof_df["id"].astype(str)
oof_df["label"] = oof_df["label"].astype(int)
oof_df["fold"] = oof_df["fold"].astype(int)
oof_df["oof_dusty_prob"] = oof_df["oof_dusty_prob"].astype(float)

oof_df["mislabel_score_oof_recomputed"] = np.where(
    oof_df["label"] == 0,
    oof_df["oof_dusty_prob"],
    1.0 - oof_df["oof_dusty_prob"],
)
score_diff = (oof_df["mislabel_score_oof"].astype(float) - oof_df["mislabel_score_oof_recomputed"]).abs().max()
print("max abs diff for mislabel_score_oof recomputation:", score_diff)
if score_diff > 1e-6:
    raise ValueError("mislabel_score_oof does not match recomputed formula")
oof_df["mislabel_score_oof"] = oof_df["mislabel_score_oof_recomputed"]
oof_df = oof_df.drop(columns=["mislabel_score_oof_recomputed"])

if "mislabel_score_v2_add" not in oof_df.columns and V2_DETAIL_PATH is not None:
    v2_df = pd.read_csv(V2_DETAIL_PATH)
    v2_df["id"] = v2_df["id"].astype(str)
    merge_cols = ["id"] + [col for col in ["mislabel_score_v2_add", "dup_score"] if col in v2_df.columns]
    oof_df = oof_df.merge(v2_df[merge_cols], on="id", how="left")

for col in ["dup_score", "mislabel_score_v2_add"]:
    if col not in oof_df.columns:
        oof_df[col] = 0.0
    oof_df[col] = oof_df[col].fillna(0.0).astype(float)

score_df = split_df[["id", "path", "label", "baseline_split"]].merge(
    oof_df[["id", "fold", "oof_dusty_prob", "oof_pred", "mislabel_score_oof", "dup_score", "mislabel_score_v2_add"]],
    on="id",
    how="left",
)
required_score_cols = ["id", "label", "fold", "baseline_split", "mislabel_score_oof", "dup_score"]
if score_df[required_score_cols].isna().any().any():
    raise ValueError("OOF scores did not merge onto every train label")

score_df["fold"] = score_df["fold"].astype(int)
score_df["mislabel_rank_oof_global"] = score_df["mislabel_score_oof"].rank(method="first", ascending=False).astype(int)
score_df["rank_in_fold"] = (
    score_df.groupby("fold")["mislabel_score_oof"]
    .rank(method="first", ascending=False)
    .astype(int)
)
score_df["is_candidate_top50"] = score_df["rank_in_fold"] <= CANDIDATE_TOP50_PER_FOLD
score_df["is_candidate_top100"] = score_df["rank_in_fold"] <= CANDIDATE_TOP100_PER_FOLD
score_df["active_candidate_set"] = score_df["is_candidate_top50"]

expected_folds = sorted(score_df["fold"].unique().tolist())
if expected_folds != list(range(len(expected_folds))):
    raise ValueError(f"fold values must be contiguous from 0: {expected_folds}")

candidate_top50_df = (
    score_df.loc[score_df["is_candidate_top50"]]
    .sort_values(["fold", "rank_in_fold"])
    .reset_index(drop=True)
)
candidate_top100_df = (
    score_df.loc[score_df["is_candidate_top100"]]
    .sort_values(["fold", "rank_in_fold"])
    .reset_index(drop=True)
)
active_candidate_ids = set(candidate_top50_df["id"])

if len(candidate_top50_df) != len(expected_folds) * CANDIDATE_TOP50_PER_FOLD:
    raise ValueError("candidate_top50 row count mismatch")
if len(candidate_top100_df) != len(expected_folds) * CANDIDATE_TOP100_PER_FOLD:
    raise ValueError("candidate_top100 row count mismatch")
if candidate_top50_df["id"].duplicated().any() or candidate_top100_df["id"].duplicated().any():
    raise ValueError("candidate ids must be unique")

for name, candidate_df, expected_per_fold in [
    ("candidate_top50", candidate_top50_df, CANDIDATE_TOP50_PER_FOLD),
    ("candidate_top100", candidate_top100_df, CANDIDATE_TOP100_PER_FOLD),
]:
    fold_counts = candidate_df["fold"].value_counts().reindex(expected_folds, fill_value=0).sort_index()
    if not (fold_counts == expected_per_fold).all():
        raise ValueError(f"{name} fold balance mismatch: {fold_counts.to_dict()}")

score_df.to_csv(TABLE_DIR / "trackA_pr_ablation_top50_fold_balanced_score_table.csv", index=False)
candidate_top50_df.to_csv(TABLE_DIR / "candidate_top50_fold_balanced.csv", index=False)
candidate_top100_df.to_csv(TABLE_DIR / "candidate_top100_fold_balanced.csv", index=False)

validation_summary = pd.DataFrame([
    {"check": "oof_rows", "value": len(oof_df)},
    {"check": "oof_unique_ids", "value": oof_df["id"].nunique()},
    {"check": "oof_source", "value": oof_source},
    {"check": "mislabel_score_recompute_max_abs_diff", "value": score_diff},
    {"check": "candidate_top50_rows", "value": len(candidate_top50_df)},
    {"check": "candidate_top100_rows", "value": len(candidate_top100_df)},
])
validation_summary.to_csv(TABLE_DIR / "oof_validation_summary.csv", index=False)
display(validation_summary)

compare_rows = []
if "mislabel_score_v2_add" in score_df.columns:
    top100_candidate_ids = set(candidate_top100_df["id"])
    top100_v2_ids = set(score_df.sort_values("mislabel_score_v2_add", ascending=False).head(100)["id"])
    overlap = len(top100_candidate_ids & top100_v2_ids)
    pearson = score_df[["mislabel_score_oof", "mislabel_score_v2_add"]].corr(method="pearson").iloc[0, 1]
    spearman = score_df[["mislabel_score_oof", "mislabel_score_v2_add"]].corr(method="spearman").iloc[0, 1]
    compare_rows.append({
        "candidate": "candidate_top100_fold_balanced",
        "v2_top100_overlap": overlap,
        "v2_top100_overlap_ratio": overlap / 100,
        "pearson_corr": pearson,
        "spearman_corr": spearman,
    })

compare_df = pd.DataFrame(compare_rows)
compare_df.to_csv(TABLE_DIR / "oof_vs_v2_compare.csv", index=False)
print("OOF fold-balanced candidate vs v2 comparison:")
display(compare_df)
print("saved score table:", TABLE_DIR / "trackA_pr_ablation_top50_fold_balanced_score_table.csv")


max abs diff for mislabel_score_oof recomputation: 0.0


,check,value
0,oof_rows,1366
1,oof_unique_ids,1366
2,oof_source,rebuilt_from_best_auc_checkpoints
3,mislabel_score_recompute_max_abs_diff,0.0
4,candidate_top50_rows,50
5,candidate_top100_rows,100


OOF fold-balanced candidate vs v2 comparison:


,candidate,v2_top100_overlap,v2_top100_overlap_ratio,pearson_corr,spearman_corr
0,candidate_top100_fold_balanced,20,0.2,0.170372,0.162973


saved score table: /home/thkim0/github/AIFFEL_quest_rs/MainQuest/Quest02/outputs/trackA_pr2_fast_top50_fold_balanced_20260702_212301/tables/trackA_pr_ablation_top50_fold_balanced_score_table.csv


## 6. Build Fold-Balanced PR2 Train Set and Sample Weights

In [ ]:
train_candidates_before = score_df.loc[score_df["baseline_split"] == "train"].copy()
val_all_df = score_df.loc[score_df["baseline_split"] == "val"].copy()
val_without_candidates_df = val_all_df.loc[~val_all_df["id"].isin(active_candidate_ids)].copy()
val_candidates_only_df = val_all_df.loc[val_all_df["id"].isin(active_candidate_ids)].copy()

train_removed_df = train_candidates_before.loc[train_candidates_before["id"].isin(active_candidate_ids)].copy()
train_candidates_after = train_candidates_before.loc[~train_candidates_before["id"].isin(active_candidate_ids)].copy()

if set(train_removed_df["id"]) - active_candidate_ids:
    raise ValueError("train_removed_df contains non-candidate ids")
if val_all_df["id"].nunique() != len(val_all_df):
    raise ValueError("validation ids must be unique")
if len(val_all_df) != len(val_without_candidates_df) + len(val_candidates_only_df):
    raise ValueError("val_all must equal val_without_candidates + val_candidates_only")
if not train_removed_df["baseline_split"].eq("train").all():
    raise ValueError("only PR0 train split samples may be removed")

class_balance_rows = []
for name, df in [
    ("train_before_removal", train_candidates_before),
    ("removed_from_train_top50_fold_balanced", train_removed_df),
    ("train_after_removal", train_candidates_after),
    ("val_all", val_all_df),
    ("val_without_candidates", val_without_candidates_df),
    ("val_candidates_only", val_candidates_only_df),
]:
    counts = df["label"].value_counts().sort_index()
    for label in [0, 1]:
        class_balance_rows.append({
            "subset": name,
            "label": label,
            "count": int(counts.get(label, 0)),
            "total": int(len(df)),
        })
class_balance_df = pd.DataFrame(class_balance_rows)
class_balance_df.to_csv(TABLE_DIR / "class_balance_before_after_top50_fold_balanced.csv", index=False)
print("class balance before/after top50 fold-balanced removal:")
display(class_balance_df)


def compute_dup_sample_weight(dup_score):
    dup_score = np.asarray(dup_score, dtype=float)
    dup_score = np.clip(dup_score, 0.0, 1.0)
    weight = 1.0 - (1.0 - DUP_WEIGHT_MIN) * np.power(dup_score, DUP_WEIGHT_POWER)
    return np.clip(weight, DUP_WEIGHT_MIN, 1.0)


score_df["sample_weight_before_removal"] = compute_dup_sample_weight(score_df["dup_score"].to_numpy())
for candidate_df in [candidate_top50_df, candidate_top100_df]:
    candidate_df["sample_weight_before_removal"] = compute_dup_sample_weight(candidate_df["dup_score"].to_numpy())
    candidate_df["removed_from_train"] = candidate_df["baseline_split"].eq("train")

candidate_top50_df.to_csv(TABLE_DIR / "candidate_top50_fold_balanced.csv", index=False)
candidate_top100_df.to_csv(TABLE_DIR / "candidate_top100_fold_balanced.csv", index=False)


def build_variant_train_df(use_dup_weight):
    df = train_candidates_after.copy()
    if use_dup_weight:
        df["sample_weight"] = compute_dup_sample_weight(df["dup_score"].to_numpy())
    else:
        df["sample_weight"] = 1.0
    return df.reset_index(drop=True)

pr1_train_df = build_variant_train_df(use_dup_weight=False)
pr2_train_df = build_variant_train_df(use_dup_weight=True)

if not (pr1_train_df["sample_weight"].astype(float) == 1.0).all():
    raise ValueError("PR1 sample_weight must be all 1.0")
if not pr2_train_df["sample_weight"].between(DUP_WEIGHT_MIN, 1.0).all():
    raise ValueError("PR2 sample_weight out of expected range")

pr1_train_df.to_csv(TABLE_DIR / "PR1_top50_fold_balanced_train_rows.csv", index=False)
pr2_train_df.to_csv(TABLE_DIR / "PR2_top50_fold_balanced_train_rows.csv", index=False)
train_removed_df.to_csv(TABLE_DIR / "train_removed_top50_fold_balanced.csv", index=False)

weight_stat_rows = []
for run_name, df in [("PR1", pr1_train_df), ("PR2", pr2_train_df)]:
    sample_weight = df["sample_weight"].astype(float)
    dup_score = df["dup_score"].astype(float)
    weight_stat_rows.append({
        "run": run_name,
        "rows": len(df),
        "sum_sample_weight": float(sample_weight.sum()),
        "mean_sample_weight": float(sample_weight.mean()),
        "min_sample_weight": float(sample_weight.min()),
        "q25_sample_weight": float(sample_weight.quantile(0.25)),
        "q50_sample_weight": float(sample_weight.quantile(0.50)),
        "q75_sample_weight": float(sample_weight.quantile(0.75)),
        "max_sample_weight": float(sample_weight.max()),
        "min_dup_score": float(dup_score.min()),
        "mean_dup_score": float(dup_score.mean()),
        "max_dup_score": float(dup_score.max()),
        "q90_dup_score": float(dup_score.quantile(0.90)),
        "q99_dup_score": float(dup_score.quantile(0.99)),
    })
weight_stats_df = pd.DataFrame(weight_stat_rows)
weight_stats_df.to_csv(TABLE_DIR / "sample_weight_stats_top50_fold_balanced.csv", index=False)
print("sample weight and dup score stats; PR1 rows are reference only, PR2 is the only training target:")
display(weight_stats_df)


## 8. Training and Evaluation Helpers


In [8]:
def binary_outputs(labels_tensor, logits_tensor):
    probs = torch.softmax(logits_tensor, dim=1)[:, 1]
    preds = logits_tensor.argmax(dim=1)
    return (
        labels_tensor.detach().cpu().numpy(),
        preds.detach().cpu().numpy(),
        probs.detach().cpu().numpy(),
    )


def train_one_epoch_weighted(model, dataloader, optimizer):
    model.train()
    loss_fn = nn.CrossEntropyLoss(reduction="none")
    total_weighted_loss = 0.0
    total_weight = 0.0
    all_labels, all_preds, all_probs = [], [], []

    for images, labels, sample_weights in dataloader:
        images = images.to(device, non_blocking=PIN_MEMORY)
        labels = labels.to(device, non_blocking=PIN_MEMORY)
        sample_weights = sample_weights.to(device, non_blocking=PIN_MEMORY).float()

        logits = model(images)
        loss_raw = loss_fn(logits, labels)
        loss = (loss_raw * sample_weights).sum() / sample_weights.sum().clamp_min(1e-8)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        batch_weight = float(sample_weights.sum().item())
        total_weighted_loss += float(loss.item()) * batch_weight
        total_weight += batch_weight

        labels_np, preds_np, probs_np = binary_outputs(labels, logits)
        all_labels.extend(labels_np.tolist())
        all_preds.extend(preds_np.tolist())
        all_probs.extend(probs_np.tolist())

    return {
        "loss": total_weighted_loss / max(total_weight, 1e-8),
        "acc": float(accuracy_score(all_labels, all_preds)),
        "auc": safe_auc(all_labels, all_probs),
        "sum_sample_weight": total_weight,
        "mean_sample_weight": total_weight / max(len(dataloader.dataset), 1),
    }


def evaluate_labeled(model, dataframe, subset_name):
    if len(dataframe) == 0:
        return {
            "subset": subset_name,
            "n": 0,
            "label0": 0,
            "label1": 0,
            "loss": np.nan,
            "acc": np.nan,
            "auc": np.nan,
            "mean_dusty_prob": np.nan,
        }
    loader = make_eval_loader(dataframe, has_label=True)
    loss_fn = nn.CrossEntropyLoss(reduction="sum")
    model.eval()
    total_loss = 0.0
    all_labels, all_preds, all_probs = [], [], []
    with torch.no_grad():
        for images, labels, _ in loader:
            images = images.to(device, non_blocking=PIN_MEMORY)
            labels = labels.to(device, non_blocking=PIN_MEMORY)
            logits = model(images)
            total_loss += float(loss_fn(logits, labels).item())
            labels_np, preds_np, probs_np = binary_outputs(labels, logits)
            all_labels.extend(labels_np.tolist())
            all_preds.extend(preds_np.tolist())
            all_probs.extend(probs_np.tolist())
    label_counts = pd.Series(all_labels).value_counts().to_dict()
    return {
        "subset": subset_name,
        "n": len(all_labels),
        "label0": int(label_counts.get(0, 0)),
        "label1": int(label_counts.get(1, 0)),
        "loss": total_loss / len(all_labels),
        "acc": float(accuracy_score(all_labels, all_preds)),
        "auc": safe_auc(all_labels, all_probs),
        "mean_dusty_prob": float(np.mean(all_probs)),
    }


def evaluate_all_val_subsets(model, run_name, epoch=None):
    rows = []
    for subset_name, df in [
        ("val_all", val_all_df),
        ("val_without_candidates", val_without_candidates_df),
        ("val_candidates_only", val_candidates_only_df),
    ]:
        row = evaluate_labeled(model, df, subset_name)
        row["run"] = run_name
        row["epoch"] = epoch
        rows.append(row)
    return pd.DataFrame(rows)


def predict_test_submission(model, result_path):
    loader = make_eval_loader(test_df, has_label=False)
    model.eval()
    rows = []
    with torch.no_grad():
        for images, image_ids in loader:
            images = images.to(device, non_blocking=PIN_MEMORY)
            logits = model(images)
            probs = torch.softmax(logits, dim=1)[:, 1].detach().cpu().numpy()
            for image_id, prob in zip(image_ids, probs):
                rows.append({"id": str(image_id), "dusty_prob": float(prob)})
    pred_df = pd.DataFrame(rows)
    result_df = submission_template[["id"]].merge(pred_df, on="id", how="left")
    if result_df["dusty_prob"].isna().any():
        raise ValueError("submission has missing predictions")
    if result_df["id"].tolist() != submission_template["id"].tolist():
        raise ValueError("submission id order mismatch")
    if not result_df["dusty_prob"].between(0, 1).all():
        raise ValueError("dusty_prob must be in [0, 1]")
    result_df.to_csv(result_path, index=False)
    return result_df


def load_checkpoint_strict(model, checkpoint_path):
    checkpoint = torch.load(checkpoint_path, map_location=device)
    load_result = model.load_state_dict(checkpoint["model_state_dict"], strict=False)
    print("checkpoint:", checkpoint_path)
    print("missing keys:", load_result.missing_keys)
    print("unexpected keys:", load_result.unexpected_keys)
    if load_result.missing_keys or load_result.unexpected_keys:
        raise ValueError("checkpoint key mismatch")
    return checkpoint


## 8. Train PR2 Only

In [9]:
def train_variant(run_name, train_part_df, use_dup_weight):
    set_seed(SEED)
    run_dir = CHECKPOINT_DIR / run_name
    run_dir.mkdir(parents=True, exist_ok=True)
    checkpoint_path = run_dir / f"{run_name}_best_val_all_auc.pt"
    history_path = TABLE_DIR / f"{run_name}_history.csv"
    submission_path = SUBMISSION_DIR / f"submissionA_{run_name}.csv"

    model = build_resnet18(num_classes=2, pretrained=True).to(device)
    trainable_df = print_trainable_parameters(model, run_name)
    trainable_df.to_csv(TABLE_DIR / f"{run_name}_trainable_parameters.csv", index=False)

    if summary is not None:
        try:
            summary(
                model,
                input_size=(BATCH_SIZE, 3, 224, 224),
                col_names=("input_size", "output_size", "num_params", "trainable"),
                depth=2,
                device=device,
            )
        except Exception as e:
            print("torchinfo.summary failed:", repr(e))

    train_loader = make_train_loader(train_part_df, shuffle=True)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

    print(f"[{run_name}] train rows:", len(train_part_df))
    print(f"[{run_name}] use_dup_weight:", use_dup_weight)
    print(f"[{run_name}] effective sample size sum:", float(train_part_df["sample_weight"].sum()))
    print(f"[{run_name}] effective sample weight mean:", float(train_part_df["sample_weight"].mean()))

    history_rows = []
    best_val_all_auc = -np.inf
    best_epoch = None

    for epoch in range(1, EPOCHS + 1):
        start_time = time.time()
        train_metrics = train_one_epoch_weighted(model, train_loader, optimizer)
        val_metrics_df = evaluate_all_val_subsets(model, run_name, epoch=epoch)
        val_all_auc = val_metrics_df.loc[val_metrics_df["subset"] == "val_all", "auc"].iloc[0]
        elapsed = time.time() - start_time

        row = {
            "run": run_name,
            "epoch": epoch,
            "train_rows": len(train_part_df),
            "use_dup_weight": use_dup_weight,
            "train_loss": train_metrics["loss"],
            "train_acc": train_metrics["acc"],
            "train_auc": train_metrics["auc"],
            "train_sum_sample_weight": train_metrics["sum_sample_weight"],
            "train_mean_sample_weight": train_metrics["mean_sample_weight"],
            "elapsed_sec": elapsed,
        }
        for _, metric_row in val_metrics_df.iterrows():
            prefix = metric_row["subset"]
            row[f"{prefix}_n"] = metric_row["n"]
            row[f"{prefix}_loss"] = metric_row["loss"]
            row[f"{prefix}_acc"] = metric_row["acc"]
            row[f"{prefix}_auc"] = metric_row["auc"]
        history_rows.append(row)

        print(
            f"{run_name} epoch={epoch:02d}/{EPOCHS} | "
            f"train_loss={train_metrics['loss']:.4f} train_auc={train_metrics['auc']:.4f} | "
            f"val_all_auc={row['val_all_auc']:.4f} "
            f"val_without_candidates_auc={row['val_without_candidates_auc']:.4f} "
            f"val_candidates_only_auc={row['val_candidates_only_auc']:.4f} | {elapsed:.1f}s"
        )

        if np.isfinite(val_all_auc) and val_all_auc > best_val_all_auc:
            best_val_all_auc = float(val_all_auc)
            best_epoch = epoch
            torch.save(
                {
                    "run": run_name,
                    "epoch": epoch,
                    "model_state_dict": model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "best_val_all_auc": best_val_all_auc,
                    "config": config,
                    "use_dup_weight": use_dup_weight,
                    "active_candidate_name": ACTIVE_CANDIDATE_NAME,
                    "removed_candidate_ids": sorted(active_candidate_ids),
                },
                checkpoint_path,
            )
            print("  -> saved best checkpoint:", checkpoint_path)

    history_df = pd.DataFrame(history_rows)
    history_df.to_csv(history_path, index=False)

    checkpoint = load_checkpoint_strict(model, checkpoint_path)
    final_metrics_df = evaluate_all_val_subsets(model, run_name, epoch=checkpoint.get("epoch"))
    submission_df = predict_test_submission(model, submission_path)

    print(f"[{run_name}] best epoch:", best_epoch, "best_val_all_auc:", best_val_all_auc)
    print(f"[{run_name}] saved history:", history_path)
    print(f"[{run_name}] saved submission:", submission_path)

    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return {
        "run": run_name,
        "checkpoint_path": checkpoint_path,
        "history_path": history_path,
        "submission_path": submission_path,
        "history_df": history_df,
        "final_metrics_df": final_metrics_df,
        "submission_df": submission_df,
    }


all_metric_rows = []
submission_paths = {}
variant_results = {}
print("Fast mode: skip PR1 retraining; this notebook trains PR2 only.")

if RUN_PR2:
    variant_results["PR2"] = train_variant("PR2_top50_fold_balanced_remove_dup_weight", pr2_train_df, use_dup_weight=True)
    all_metric_rows.append(variant_results["PR2"]["final_metrics_df"])
    submission_paths["PR2"] = variant_results["PR2"]["submission_path"]
else:
    print("RUN_PR2=False; skip PR2 retraining")

if not all_metric_rows:
    raise RuntimeError("No variant was trained. RUN_PR2 must be True for this notebook.")
val_metrics_by_subset_df = pd.concat(all_metric_rows, ignore_index=True)
val_metrics_by_subset_df.to_csv(TABLE_DIR / "val_metrics_by_subset_top50_fold_balanced.csv", index=False)
print("saved val subset metrics:", TABLE_DIR / "val_metrics_by_subset_top50_fold_balanced.csv")
display(val_metrics_by_subset_df)


Fast mode: skip PR1 retraining; this notebook trains PR2 only.


NameError: name 'pr2_train_df' is not defined

## 9. Final Artifact Checklist

In [ ]:
artifact_paths = [
    CONFIG_PATH,
    TABLE_DIR / "pr0_split_assignments.csv",
    TABLE_DIR / "PR2_top50_fold_balanced_train_rows.csv",
    TABLE_DIR / "PR2_top50_fold_balanced_remove_dup_weight_history.csv",
    SUBMISSION_DIR / "submissionA_PR2_top50_fold_balanced_remove_dup_weight.csv",
    TABLE_DIR / "oof_validation_summary.csv",
    TABLE_DIR / "oof_checkpoint_audit.csv",
    TABLE_DIR / "oof_vs_v2_compare.csv",
    TABLE_DIR / "trackA_pr_ablation_top50_fold_balanced_score_table.csv",
    TABLE_DIR / "candidate_top50_fold_balanced.csv",
    TABLE_DIR / "candidate_top100_fold_balanced.csv",
    TABLE_DIR / "class_balance_before_after_top50_fold_balanced.csv",
    TABLE_DIR / "sample_weight_stats_top50_fold_balanced.csv",
    TABLE_DIR / "val_metrics_by_subset_top50_fold_balanced.csv",
]

checklist_df = pd.DataFrame({
    "path": [str(path) for path in artifact_paths],
    "exists": [Path(path).exists() for path in artifact_paths],
})
checklist_df.to_csv(TABLE_DIR / "artifact_checklist_pr2_fast_top50_fold_balanced.csv", index=False)
display(checklist_df)

print("OUTPUT_DIR:", OUTPUT_DIR)
print("Active candidate set:", ACTIVE_CANDIDATE_NAME)
print("Fast notebook mode: PR2 only, with top50 fold-balanced removal + duplicate down-weighting.")
